# Reproducibility smoke test: 400-patient sample

This notebook runs `../validation/benchmark_script.py`, **unmodified, the exact script used for the published benchmark**, against the 400-patient sample in `../sample`.

**The published numbers come from the full 20,000-patient cohort** (5-fold stratified cross-validation, 16,000 train / 4,000 test per fold, seed 20261119): XGBoost with characteristics, first level and microbiology reaches AUROC 0.970 (SD 0.003); without microbiology 0.809 (SD 0.005). Those are the only numbers to quote. On 400 patients (about 100 per test fold) every score below is noisy and must not be quoted.

In [1]:
import subprocess, sys
r = subprocess.run([sys.executable, "../validation/benchmark_script.py", "../sample", "./smoke_test_results.parquet"],
                   capture_output=True, text=True)
print(r.stdout[-3000:])
print(r.stderr[-2000:])
assert r.returncode == 0, r.stderr

                                                                         auroc                   auprc                   brier                
                                                                          mean     std     min    mean     std     min    mean     std     min
feature_set                                        model                                                                                      
characteristics_plus_first_level                   logistic_regression  0.8055  0.0557  0.7191  0.6359  0.1050  0.4690  0.1429  0.0219  0.1162
                                                   xgboost              0.7873  0.0527  0.7191  0.5897  0.0589  0.5208  0.1561  0.0220  0.1326
characteristics_plus_first_level_plus_microbiology logistic_regression  0.9327  0.0382  0.8741  0.8643  0.0695  0.7590  0.0904  0.0274  0.0561
                                                   xgboost              0.9478  0.0291  0.8975  0.8912  0.0659  0.7755  0.0809  0.0323  0.0532

## Reading the output

- Each row of the results is one fold of one model on one feature set. With 400 patients the standard deviation across folds is around 0.03 to 0.05, ten times the published one, so differences between models at this size are not meaningful.
- What the sample does show is the shape: microbiology lifts the score, the first drug level adds the patient's own clearance, and the pipeline runs end to end on the same tables, columns and keys as the full cohort.
- Judge methods on the full cohort.

In [2]:
import pandas as pd
res = pd.read_parquet("./smoke_test_results.parquet")
res.groupby(["feature_set", "model"])[["auroc", "auprc", "brier"]].agg(["mean", "std"]).round(3)

auroc  \
                                                                         mean   
feature_set                                        model                        
characteristics_plus_first_level                   logistic_regression  0.805   
                                                   xgboost              0.787   
characteristics_plus_first_level_plus_microbiology logistic_regression  0.933   
                                                   xgboost              0.948   
characteristics_plus_microbiology                  logistic_regression  0.861   
                                                   xgboost              0.870   

                                                                               \
                                                                          std   
feature_set                                        model                        
characteristics_plus_first_level                   logistic_regression  0.056   
                                                   xgboost              0.053   
characteristics_plus_first_level_plus_microbiology logistic_regression  0.038   
                                                   xgboost              0.029   
characteristics_plus_microbiology                  logistic_regression  0.051   
                                                   xgboost              0.052   

                                                                        auprc  \
                                                                         mean   
feature_set                                        model                        
characteristics_plus_first_level                   logistic_regression  0.636   
                                                   xgboost              0.590   
characteristics_plus_first_level_plus_microbiology logistic_regression  0.864   
                                                   xgboost              0.891   
characteristics_plus_microbiology                  logistic_regression  0.754   
                                                   xgboost              0.740   

                                                                               \
                                                                          std   
feature_set                                        model                        
characteristics_plus_first_level                   logistic_regression  0.105   
                                                   xgboost              0.059   
characteristics_plus_first_level_plus_microbiology logistic_regression  0.070   
                                                   xgboost              0.066   
characteristics_plus_microbiology                  logistic_regression  0.091   
                                                   xgboost              0.107   

                                                                        brier  \
                                                                         mean   
feature_set                                        model                        
characteristics_plus_first_level                   logistic_regression  0.143   
                                                   xgboost              0.156   
characteristics_plus_first_level_plus_microbiology logistic_regression  0.090   
                                                   xgboost              0.081   
characteristics_plus_microbiology                  logistic_regression  0.121   
                                                   xgboost              0.118   

                                                                               
                                                                          std  
feature_set                                        model                       
characteristics_plus_first_level                   logistic_regression  0.022  
                                                   xgboost              0.022  
characteristics_plus_f